# Phase 1, Week 2: 5-core ratings, book metadata, genre-slice decision

Each Colab notebook has its own runtime, so the first four cells repeat `00_colab_setup`
(mount Drive, clone, install without torch). `data.download` keeps the 5-core ratings CSV
(0.14 GB) on Drive; `data.meta` streams `meta_Books` (4.94 GB) from the UCSD server and filters it
on the fly, so nothing large is stored.

**While it runs:** `data.meta` takes a while (the whole 4.94 GB file passes through) and prints a
progress line about once a minute with GB done, speed and time left. Colab shows an empty text box
under a running `!` command; it is Colab's input box, not a question. Leave it empty and do not
stop the cell. If Colab disconnects anyway, run the setup cells and the same cell again: work
saved in the checkpoints on Drive is reused.

Outputs in `MyDrive/litsense/data/v0/`: `ratings_5core.csv.gz`, `books_raw.parquet`,
`category_counts.csv`, `genre_slice_proposal.json`; run manifests in `MyDrive/litsense/manifests/v0/`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
REPO = "https://github.com/Aniketdhar810/LitSense.git"
BRANCH = "main"  # the branch to run; change it to test a branch under review
!rm -rf /content/LitSense
!git clone --depth 1 --branch $BRANCH $REPO /content/LitSense

In [ ]:
%cd /content/LitSense
!make install-colab

In [ ]:
!python --version
!pip list 2>/dev/null | grep -iE "^(torch|numpy|pandas|pyarrow|requests) "
!python -m ml.stages list

In [ ]:
%cd /content/LitSense
!python -m ml.stages run all --until data.meta < /dev/null

In [ ]:
!python -m ml.data.meta

**Stop here.** Choose `data.genre_slices` from the report above (`category_counts.csv` has the
full table) and send the choice back; it is recorded in `ml/config.yaml` and `docs/decisions.md`
before any later stage runs.

Optional, once `data.use_goodreads_graph: true` is committed: rerun the setup cells above (to
pull it), then the cell below to build the Goodreads ISBN index (streams about 2.2 GB, keeps only
matching rows).

In [ ]:
%cd /content/LitSense
!python -m ml.stages run data.goodreads < /dev/null